In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import rioxarray as rxr
import cartopy.crs as ccrs

In [ ]:
from pathlib import Path

rlut_path = Path("../../data/CanESM5_1850-2100_rlut.nc")
rlutcs_path = Path("../../data/CanESM_1850-2100_rlutcs.nc")
output_path = Path("../../data/CanESM_1850-2100_rlutcre.nc")

rlut_ds = xr.open_dataset(rlut_path, engine="netcdf4")
rlutcs_ds = xr.open_dataset(rlutcs_path, engine="netcdf4")

In [ ]:

rlut_var = "rlut" if "rlut" in rlut_ds.data_vars else list(rlut_ds.data_vars)[0]
rlutcs_var = "rlutcs" if "rlutcs" in rlutcs_ds.data_vars else list(rlutcs_ds.data_vars)[0]

rlut_da = rlut_ds[rlut_var]
rlutcs_da = rlutcs_ds[rlutcs_var]

rlut_da, rlutcs_da = xr.align(rlut_da, rlutcs_da, join="inner")

ESM5_data = xr.merge([
    rlut_da.to_dataset(name="rlut"),
    rlutcs_da.to_dataset(name="rlutcs"),
])

lwcre = ESM5_data["rlut"] - ESM5_data["rlutcs"]

if "member" in lwcre.dims:
    lwcre = lwcre.mean("member")

lwcre_ds = lwcre.to_dataset(name="cre")
lwcre_ds["cre"].attrs["long_name"] = "Longwave Cloud Radiative Effect"
lwcre_ds["cre"].attrs["description"] = "Computed as rlut - rlutcs"

lwcre_ds.to_netcdf(output_path)

print(f"Built CRE dataset: {output_path}")
print(lwcre_ds)

Built CRE dataset: ..\..\data\CanESM_1850-2100_rsutcre.nc
<xarray.Dataset> Size: 99MB
Dimensions:  (time: 3012, lat: 64, lon: 128)
Coordinates:
  * time     (time) object 24kB 1850-01-16 12:00:00 ... 2100-12-16 12:00:00
  * lat      (lat) float64 512B -87.86 -85.1 -82.31 -79.53 ... 82.31 85.1 87.86
  * lon      (lon) float64 1kB 0.0 2.812 5.625 8.438 ... 348.8 351.6 354.4 357.2
Data variables:
    cre      (time, lat, lon) float32 99MB 9.912 9.971 10.03 ... 0.0 0.0 0.0
